In [0]:

df = spark.read.parquet(
    "/Volumes/workspace/default/nyc_taxi_raw/yellow_tripdata_2024-01.parquet"
)

display(df.limit(10))


In [0]:

df.printSchema()


In [0]:

df.count()


In [0]:

from pyspark.sql.functions import col

df.filter(col("passenger_count").isNull()).count()


In [0]:

df.filter(col("trip_distance") <= 0).count()


In [0]:

df.filter(col("total_amount") < 0).count()


In [0]:

display(
    df.filter(col("total_amount") < 0)
      .select(
          "trip_distance",
          "fare_amount",
          "tip_amount",
          "total_amount",
          "payment_type"
      )
      .limit(10)
)


In [0]:

df.filter(col("total_amount") < 0) \
  .groupBy("payment_type") \
  .count() \
  .orderBy("payment_type") \
  .show()


In [0]:

df.filter(col("total_amount").isNull()).count()


In [0]:

df.filter(
    col("tpep_dropoff_datetime") < col("tpep_pickup_datetime")
).count()


In [0]:

df.filter(
    col("tpep_dropoff_datetime") == col("tpep_pickup_datetime")
).count()


In [0]:

display(
    df.filter(
        col("tpep_dropoff_datetime") == col("tpep_pickup_datetime")
    )
    .select(
        "trip_distance",
        "fare_amount",
        "total_amount"
    )
    .limit(10)
)


In [0]:

df.filter(
    (col("tpep_dropoff_datetime") == col("tpep_pickup_datetime")) &
    (col("trip_distance") == 0)
).count()


In [0]:

display(
    df.filter(
        (col("tpep_dropoff_datetime") == col("tpep_pickup_datetime")) &
        (col("trip_distance") != 0)
    )
    .select(
        "trip_distance",
        "fare_amount",
        "total_amount"
    )
    .limit(10)
)


In [0]:

total = df.count()
unicos = df.dropDuplicates().count()

print("Total de registros:", total)
print("Registros únicos:", unicos)
print("Possíveis duplicados:", total - unicos)


In [0]:

df.filter(col("trip_distance").isNull()).count()


In [0]:

df.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("workspace.default.bronze_nyc_taxi")


In [0]:

spark.catalog.tableExists("workspace.default.bronze_nyc_taxi")


In [0]:

df.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("workspace.default.bronze_nyc_taxi")


In [0]:

spark.table("workspace.default.bronze_nyc_taxi").count()


In [0]:

df_silver = spark.table("workspace.default.bronze_nyc_taxi")


In [0]:

from pyspark.sql.functions import when

df_silver = df_silver.withColumn(
    "status_distancia",
    when(col("trip_distance") < 0, "DISTANCIA_NEGATIVA")
    .when(col("trip_distance") == 0, "DISTANCIA_ZERO")
    .otherwise("VALIDA")
)


In [0]:

df_silver.groupBy("status_distancia").count().show()


In [0]:

df_silver = df_silver.withColumn(
    "status_passageiros",
    when(col("passenger_count").isNull(), "NAO_INFORMADO")
    .otherwise("INFORMADO")
)


In [0]:

df_silver.groupBy("status_passageiros").count().show()


In [0]:

df_silver = df_silver.withColumn(
    "status_horario",
    when(
        col("tpep_dropoff_datetime") < col("tpep_pickup_datetime"),
        "HORARIO_INVERTIDO"
    )
    .when(
        col("tpep_dropoff_datetime") == col("tpep_pickup_datetime"),
        "DURACAO_ZERO"
    )
    .otherwise("HORARIO_VALIDO")
)


In [0]:

df_silver.groupBy("status_horario").count().show()


In [0]:

df_silver = df_silver.withColumn(
    "status_financeiro",
    when(col("total_amount").isNull(), "VALOR_NULO")
    .when(col("total_amount") < 0, "VALOR_NEGATIVO")
    .when(col("total_amount") == 0, "VALOR_ZERO")
    .otherwise("VALOR_POSITIVO")
)


In [0]:

df_silver.groupBy("status_financeiro").count().show()


In [0]:

df_silver.select(
    "status_distancia",
    "status_passageiros",
    "status_horario",
    "status_financeiro"
).show(10, truncate=False)


In [0]:

df_silver.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("workspace.default.silver_nyc_taxi")


In [0]:

spark.table("workspace.default.silver_nyc_taxi").count()


In [0]:

spark.table("workspace.default.silver_nyc_taxi").printSchema()


In [0]:

df_gold = spark.table("workspace.default.silver_nyc_taxi")


In [0]:

df_gold.filter(
    (col("status_financeiro") == "VALOR_POSITIVO") &
    (col("status_horario") == "HORARIO_VALIDO")
).count()


In [0]:

from pyspark.sql.functions import sum as spark_sum

df_gold.filter(
    (col("status_financeiro") == "VALOR_POSITIVO") &
    (col("status_horario") == "HORARIO_VALIDO")
).agg(
    spark_sum("total_amount").alias("valor_total_positivo")
).show()


In [0]:

from pyspark.sql.functions import avg

df_gold.filter(
    (col("status_financeiro") == "VALOR_POSITIVO") &
    (col("status_horario") == "HORARIO_VALIDO")
).agg(
    avg("total_amount").alias("valor_medio_corrida")
).show()


In [0]:

df_gold.filter(
    (col("status_distancia") == "VALIDA") &
    (col("status_financeiro") == "VALOR_POSITIVO") &
    (col("status_horario") == "HORARIO_VALIDO")
).agg(
    avg("trip_distance").alias("distancia_media_milhas")
).show()


In [0]:

from pyspark.sql.functions import hour

df_gold = df_gold.withColumn(
    "hora_embarque",
    hour(col("tpep_pickup_datetime"))
)


In [0]:

df_gold.filter(
    (col("status_financeiro") == "VALOR_POSITIVO") &
    (col("status_horario") == "HORARIO_VALIDO")
).groupBy("hora_embarque") \
 .count() \
 .orderBy("hora_embarque") \
 .show(24)


In [0]:

df_gold_horarios.count()


In [0]:

from pyspark.sql.functions import col, hour

df_gold_horarios = (
    spark.table("workspace.default.silver_nyc_taxi")
    .filter(
        (col("status_financeiro") == "VALOR_POSITIVO") &
        (col("status_horario") == "HORARIO_VALIDO")
    )
    .withColumn(
        "hora_embarque",
        hour(col("tpep_pickup_datetime"))
    )
    .groupBy("hora_embarque")
    .count()
    .orderBy("hora_embarque")
)


In [0]:

df_gold_horarios.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("workspace.default.gold_nyc_taxi_horarios")


In [0]:

spark.table("workspace.default.gold_nyc_taxi_horarios").count()


In [0]:

from pyspark.sql.functions import col, hour, sum as spark_sum

df_gold_faturamento = (
    spark.table("workspace.default.silver_nyc_taxi")
    .filter(
        (col("status_financeiro") == "VALOR_POSITIVO") &
        (col("status_horario") == "HORARIO_VALIDO")
    )
    .withColumn(
        "hora_embarque",
        hour(col("tpep_pickup_datetime"))
    )
    .groupBy("hora_embarque")
    .agg(
        spark_sum("total_amount").alias("valor_total")
    )
    .orderBy("hora_embarque")
)


In [0]:

df_gold_faturamento.show(24, truncate=False)


In [0]:

df_gold_faturamento.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("workspace.default.gold_nyc_taxi_faturamento")


In [0]:

spark.table("workspace.default.gold_nyc_taxi_faturamento").count()


In [0]:

display(
    spark.table("workspace.default.gold_nyc_taxi_horarios")
)


In [0]:

display(
    spark.table("workspace.default.gold_nyc_taxi_faturamento")
)
